# Section VII Applications Evidence Lab (Full-Scan + JSON/Markdown Fusion)

This notebook builds the evidence layer for Section VII (Applications and Use Cases).
- Full scan: processed markdowns + O_ISAC JSON
- Variant-aware retrieval (lexical + fuzzy + LLM entailment)
- Two-model flow:
  - Pass-1 model (fast): broad classification over all hits
  - Pass-2 model (strict): only escalated uncertain hits
- Outputs: application evidence tables, transfer map, and COMST-ready synthesis artifacts

Usage note:
- Tune model names and per-model RPM in `# @title 3. Config`.
- Keep `RESUME=True`; checkpoints are under `analysis/VII_ev_v2/checkpoints`.


In [1]:
# @title 1. Install Dependencies
!pip install -q groq rapidfuzz tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.0 MB/s eta 0:00:00


In [2]:
# @title 2. Setup & Mount Drive
from google.colab import drive, userdata
import os, re, json, glob
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print('Working dir:', os.getcwd())
else:
    print('Path not found:', BASE_DIR)


Mounted at /content/drive
Working dir: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST


In [3]:
# @title 3. Config
PROCESSED_MD_DIR = Path('data/proc_markdowns')
JSON_DIR = Path('data/ext_res_v4')
UNIFIED_JSON = JSON_DIR / 'extraction_v4_unified.json'
OUTPUT_DIR = Path('analysis/VII_ev_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PROFILE = 'FULL_RESCAN'
NORMALIZE_LABELS = True
TARGET_PAPERS = None
LIMIT = None
LLM_CALLS = True

MODEL_PASS1 = 'meta-llama/llama-4-scout-17b-16e-instruct'
MODEL_PASS2 = 'llama-3.3-70b-versatile'
MODEL_VARIANT_GEN = MODEL_PASS1
USE_ESCALATION = True
ESCALATE_LABELS = {'INDIRECT', 'NONE', 'WEAK'}

RPM_BY_MODEL = {MODEL_PASS1: 120, MODEL_PASS2: 40, MODEL_VARIANT_GEN: 120}
DEFAULT_RPM = 30
MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0

MAX_VARIANTS_PER_CONCEPT = 12
MAX_HITS_PER_CONCEPT_PER_PAPER = 6
MAX_CONTEXT_CHARS = 1200
CLASSIFY_CHUNK_SIZE = 4
BATCH_SIZE_PAPERS = 10

RESUME = True
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_MAP_PATH = Path('analysis/II_sch_map.md')
GOV_PATH = Path('analysis/II_met_gov.md')
IV_AXIS_PATH = Path('analysis/IV_ev_v2/axis_definitions.md')
IV_MAP_PATH = Path('analysis/IV_ev_v2/mapping_rules.md')
VI_AXIS_PATH = Path('analysis/VI_ev_v2/axis_definitions.md')
VI_MAP_PATH = Path('analysis/VI_ev_v2/mapping_rules.md')

schema_text = SCHEMA_MAP_PATH.read_text(encoding='utf-8', errors='ignore') if SCHEMA_MAP_PATH.exists() else ''
gov_text = GOV_PATH.read_text(encoding='utf-8', errors='ignore') if GOV_PATH.exists() else ''
iv_axis_text = IV_AXIS_PATH.read_text(encoding='utf-8', errors='ignore') if IV_AXIS_PATH.exists() else ''
iv_map_text = IV_MAP_PATH.read_text(encoding='utf-8', errors='ignore') if IV_MAP_PATH.exists() else ''
vi_axis_text = VI_AXIS_PATH.read_text(encoding='utf-8', errors='ignore') if VI_AXIS_PATH.exists() else ''
vi_map_text = VI_MAP_PATH.read_text(encoding='utf-8', errors='ignore') if VI_MAP_PATH.exists() else ''

print('Config ready. Output:', OUTPUT_DIR)
print('Run profile:', RUN_PROFILE)
print('Pass-1 model:', MODEL_PASS1)
print('Pass-2 model:', MODEL_PASS2)
print('Section II schema loaded:', bool(schema_text))
print('Section II governance loaded:', bool(gov_text))
print('Section IV axis loaded:', bool(iv_axis_text))
print('Section IV mapping loaded:', bool(iv_map_text))
print('Section VI axis loaded:', bool(vi_axis_text))
print('Section VI mapping loaded:', bool(vi_map_text))


Config ready. Output: analysis/VII_ev_v2
Run profile: FULL_RESCAN
Pass-1 model: meta-llama/llama-4-scout-17b-16e-instruct
Pass-2 model: llama-3.3-70b-versatile
Section II schema loaded: True
Section II governance loaded: True
Section IV axis loaded: True
Section IV mapping loaded: True
Section VI axis loaded: True
Section VI mapping loaded: True


In [4]:
# @title 4. Load O_ISAC JSON Index
def load_json_index(json_dir: Path):
    index = {}
    for p in sorted(json_dir.glob('O_ISAC_*_v4.json')):
        paper_id = p.stem.replace('_v4','')
        try:
            index[paper_id] = json.loads(p.read_text(encoding='utf-8', errors='ignore'))
        except Exception as e:
            index[paper_id] = {'_error': str(e)}
    unified = None
    if UNIFIED_JSON.exists():
        unified = json.loads(UNIFIED_JSON.read_text(encoding='utf-8', errors='ignore'))
    return index, unified

json_index, unified_json = load_json_index(JSON_DIR)
print('JSON files loaded:', len(json_index))
print('Unified JSON:', 'yes' if unified_json else 'no')


JSON files loaded: 221
Unified JSON: yes


In [5]:
# @title 5. Load Processed Markdowns (Canonical per paper)
def canonical_md_path(paths, paper_id):
    scored = []
    for p in paths:
        p = Path(p)
        score = 0
        if (p.parent / 'visual_analysis.txt').exists():
            score += 3
        if p.parent.name == paper_id and p.parent.parent.name == paper_id:
            score += 2
        score += len(p.parts) * 0.1
        scored.append((score, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored else None

def load_processed_markdowns(target_ids=None, limit=None):
    search_path = PROCESSED_MD_DIR
    all_files = list(search_path.rglob('*.md'))
    md_files = [p for p in all_files if 'O_ISAC_' in p.name]

    grouped = {}
    for p in md_files:
        m = re.search(r'(O_ISAC_\d+)', p.name)
        if not m:
            continue
        paper_id = m.group(1)
        if target_ids and paper_id not in target_ids:
            continue
        grouped.setdefault(paper_id, []).append(p)

    records = []
    for i, (paper_id, paths) in enumerate(sorted(grouped.items())):
        if limit and i >= limit:
            break
        canon = canonical_md_path(paths, paper_id)
        if not canon:
            continue
        text = canon.read_text(encoding='utf-8', errors='ignore')
        lines = text.splitlines()
        va_path = canon.parent / 'visual_analysis.txt'
        va_text = va_path.read_text(encoding='utf-8', errors='ignore') if va_path.exists() else ''
        records.append({
            'paper_id': paper_id,
            'md_path': str(canon),
            'text': text,
            'lines': lines,
            'visual_analysis': va_text
        })
    return records

papers = load_processed_markdowns(target_ids=TARGET_PAPERS, limit=LIMIT)
print('Markdown papers loaded:', len(papers))


Markdown papers loaded: 221


In [6]:
# @title 6. Heading + Application Helpers
import re

MEDIUM_ALIAS_MAP = {
    'visible_light': 'wireless_vlc',
    'vlc': 'wireless_vlc',
    'rf': 'wireless_rf',
    'photo_thz': 'terahertz',
    'photonic_thz': 'terahertz',
}

APP_DOMAIN_ALIAS = {
    'indoor_localization': 'indoor_positioning',
    'vehicular_networks': 'vehicular',
    '6g': '6g_networks',
    'leo_satellite_isac': 'satellite_communication',
    'space': 'aerospace_space',
    'space_sustainability': 'aerospace_space',
    'subsea_monitoring': 'maritime_underwater',
    'underwater_surveillance': 'maritime_underwater',
    'uav_communication': 'uav_aerial',
    'industrial_iot': 'industrial_manufacturing',
    'optical_network_monitoring': 'fibre_network_monitoring',
    'optical_fiber_communications': 'fibre_network_monitoring',
    'submarine_fiber_optical_networks': 'fibre_network_monitoring',
    'smart_cities': 'critical_infrastructure',
}

MACRO_DOMAIN_RULES = {
    'smart_infrastructure': {
        'domains': {'industrial_manufacturing', 'critical_infrastructure', 'fibre_network_monitoring', 'environmental_monitoring', 'power_grid', 'distributed_sensing', 'telecommunication', 'optical_communication', 'optical_networks', 'iot', 'digital_twin', 'security_surveillance', 'geophysics', 'data_center', 'short_reach_optical_interconnects', 'logistics'},
        'keywords': ['infrastructure', 'pipeline', 'structural health', 'grid', 'factory', 'industry', 'monitoring network'],
    },
    'indoor_environments': {
        'domains': {'indoor_positioning', 'healthcare', 'human_computer_interaction', 'assisted_living', 'personnel_monitoring', 'asset_tracking', 'medical_imaging', 'biochemical_sensing'},
        'keywords': ['indoor', 'hospital', 'smart lighting', 'retail', 'room', 'building', 'occupancy'],
    },
    'automotive_transportation': {
        'domains': {'vehicular', 'autonomous_vehicles', 'intelligent_transportation_systems', 'traffic_monitoring', 'uav_aerial', 'robotics_autonomy'},
        'keywords': ['vehicular', 'v2v', 'v2i', 'automotive', 'transport', 'traffic', 'driving', 'uav'],
    },
    'underwater_harsh': {
        'domains': {'maritime_underwater', 'oceanography', 'volcanic_ash_detection', 'nondestructive_testing', 'concentrated_solar_power', 'microalgae_cultivation'},
        'keywords': ['underwater', 'subsea', 'ocean', 'marine', 'harsh', 'extreme', 'volcanic', 'corrosive'],
    },
    'space_satellite': {
        'domains': {'aerospace_space', 'satellite_communication', 'quantum_communication'},
        'keywords': ['satellite', 'inter-satellite', 'space', 'orbital', 'leo', 'debris', 'aerospace'],
    },
}


def build_heading_map(lines):
    current = []
    heading_map = {}
    for i, line in enumerate(lines):
        if line.startswith('#'):
            level = len(line) - len(line.lstrip('#'))
            title = line.strip('#').strip()
            if level <= len(current):
                current = current[:level - 1]
            current.append(title)
        heading_map[i] = ' > '.join(current) if current else 'no_heading'
    return heading_map


def get_context(lines, idx, window=2):
    start = max(0, idx - window)
    end = min(len(lines), idx + window + 1)
    return '\\n'.join(lines[start:end])


def normalize_token(value):
    s = str(value or '').strip().lower()
    if not s or s in {'nr', 'not reported', 'nan', 'none', 'na', 'n/a'}:
        return 'unknown'
    s = s.replace('/', '_').replace('&', ' and ')
    s = re.sub(r'[^a-z0-9_\-\s]+', '', s)
    s = re.sub(r'[\s\-]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s or 'unknown'


def normalize_medium_label(value):
    s = normalize_token(value)
    return MEDIUM_ALIAS_MAP.get(s, s)


def get_record_medium(record):
    if not isinstance(record, dict):
        return 'unknown'
    clsf = record.get('study_level', {}).get('classification', {})
    if not isinstance(clsf, dict):
        return 'unknown'
    return normalize_medium_label(clsf.get('oisac_medium_class', 'unknown'))


def get_scenarios(record):
    if not isinstance(record, dict):
        return []
    sl = record.get('scenario_level', [])
    if isinstance(sl, list):
        return [x for x in sl if isinstance(x, dict)]
    if isinstance(sl, dict):
        return [sl]
    return []


def as_list(value):
    if isinstance(value, list):
        return value
    if value is None:
        return []
    return [value]


def get_application_domains_raw(record):
    if not isinstance(record, dict):
        return []
    app = record.get('study_level', {}).get('application', {})
    if not isinstance(app, dict):
        return []
    out = []
    seen = set()
    for v in as_list(app.get('application_domain')):
        tok = normalize_token(v)
        if tok != 'unknown' and tok not in seen:
            seen.add(tok)
            out.append(tok)
    return out


def get_application_description(record):
    if not isinstance(record, dict):
        return ''
    app = record.get('study_level', {}).get('application', {})
    return str(app.get('scenario_description', '') or '').strip() if isinstance(app, dict) else ''


def get_scenario_labels(record):
    labels = []
    for scn in get_scenarios(record):
        ident = scn.get('identification', {}) if isinstance(scn, dict) else {}
        lab = ident.get('scenario_label', '') if isinstance(ident, dict) else ''
        if not lab and isinstance(scn, dict):
            lab = scn.get('scenario_label', '')
        lab = str(lab).strip()
        if lab:
            labels.append(lab)
    out = []
    seen = set()
    for x in labels:
        k = x.lower()
        if k not in seen:
            seen.add(k)
            out.append(x)
    return out


def get_application_domains_canonical(record):
    out = []
    seen = set()
    for d in get_application_domains_raw(record):
        c = APP_DOMAIN_ALIAS.get(d, d)
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out


def infer_macro_domains(canonical_domains, scenario_description=''):
    macros = set()
    desc = str(scenario_description or '').lower()
    for dom in canonical_domains:
        for macro, rule in MACRO_DOMAIN_RULES.items():
            if dom in rule['domains']:
                macros.add(macro)
    for macro, rule in MACRO_DOMAIN_RULES.items():
        if any(k in desc for k in rule['keywords']):
            macros.add(macro)
    return sorted(macros)


def get_application_bundle(record):
    raw_domains = get_application_domains_raw(record)
    canonical_domains = get_application_domains_canonical(record)
    scenario_description = get_application_description(record)
    scenario_labels = get_scenario_labels(record)
    macro_domains = infer_macro_domains(canonical_domains, scenario_description)
    return {
        'raw_domains': raw_domains,
        'canonical_domains': canonical_domains,
        'macro_domains': macro_domains,
        'scenario_description': scenario_description,
        'scenario_labels': scenario_labels,
    }


def get_macro_domains_from_json(record):
    b = get_application_bundle(record)
    return b['macro_domains']



In [7]:
# @title 7. Groq Client + Variant Generator (Cache + Per-Model Rate Limit)
from groq import Groq
from collections import deque
import time
import random

VARIANT_CACHE = OUTPUT_DIR / 'variant_cache.json'
if VARIANT_CACHE.exists():
    variant_cache = json.loads(VARIANT_CACHE.read_text(encoding='utf-8'))
else:
    variant_cache = {}

_GROQ_CLIENT = None
REQUEST_LOG_BY_MODEL = {}


def get_groq_client():
    global _GROQ_CLIENT
    if _GROQ_CLIENT is not None:
        return _GROQ_CLIENT

    try:
        api_key = userdata.get('GROQ_API_KEY')
    except Exception:
        api_key = os.environ.get('GROQ_API_KEY')

    if not api_key:
        raise ValueError('GROQ_API_KEY not found in Colab Secrets or env.')

    _GROQ_CLIENT = Groq(api_key=api_key)
    return _GROQ_CLIENT


def get_model_rpm(model_name):
    return RPM_BY_MODEL.get(model_name, DEFAULT_RPM)


def throttle_requests(model_name):
    rpm = get_model_rpm(model_name)
    if rpm <= 0:
        return

    q = REQUEST_LOG_BY_MODEL.setdefault(model_name, deque())
    now = time.time()

    while q and now - q[0] > 60:
        q.popleft()

    if len(q) >= rpm:
        wait_s = 60 - (now - q[0]) + 0.1
        wait_s = max(wait_s, 0.1)
        print(f'Rate limit guard ({model_name}): sleeping {wait_s:.1f}s')
        time.sleep(wait_s)
        now = time.time()
        while q and now - q[0] > 60:
            q.popleft()

    q.append(time.time())


def safe_chat_completion(model_name, messages, expect_json=False, temperature=0.2):
    if not LLM_CALLS:
        return None

    client = get_groq_client()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            throttle_requests(model_name)
            kwargs = {
                'model': model_name,
                'messages': messages,
                'temperature': temperature
            }
            if expect_json:
                kwargs['response_format'] = {'type': 'json_object'}

            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            if attempt >= MAX_RETRIES:
                print(f'LLM call failed ({model_name}) after {MAX_RETRIES} attempts: {e}')
                return None
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1)) + random.uniform(0.0, 0.5)
            print(f'LLM retry ({model_name}) {attempt}/{MAX_RETRIES}: {e}; sleeping {sleep_s:.1f}s')
            time.sleep(sleep_s)


def get_variants(concept):
    if concept in variant_cache:
        vals = variant_cache[concept]
        return vals[:MAX_VARIANTS_PER_CONCEPT]

    if not LLM_CALLS:
        vals = [concept]
        variant_cache[concept] = vals
        return vals

    system_prompt = (
        'You generate lexical variants and paraphrases for evidence retrieval. '
        'Return compact JSON: {"variants": ["..."]}.'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        'Return up to 12 variants including synonyms, abbreviations, paraphrases, and morphological forms. '
        'Keep each variant short.'
    )

    content = safe_chat_completion(
        model_name=MODEL_VARIANT_GEN,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.2
    )

    variants = [concept]
    if content:
        try:
            data = json.loads(content)
            llm_vars = data.get('variants', [])
            if isinstance(llm_vars, list):
                for item in llm_vars:
                    if isinstance(item, str) and item.strip():
                        variants.append(item.strip())
        except Exception:
            pass

    dedup = []
    seen = set()
    for v in variants:
        key = v.lower().strip()
        if not key or key in seen:
            continue
        seen.add(key)
        dedup.append(v)

    dedup = dedup[:MAX_VARIANTS_PER_CONCEPT]
    variant_cache[concept] = dedup
    VARIANT_CACHE.write_text(json.dumps(variant_cache, ensure_ascii=False, indent=2), encoding='utf-8')
    return dedup


In [8]:
# @title 8. Retrieval + Entailment Classification (Two-Model, Batched, Checkpointed)
def scan_lines_for_variants(lines, variants, fuzzy_threshold=85):
    hits = []
    for i, line in enumerate(lines):
        text = line.strip()
        if not text:
            continue
        low = text.lower()
        for v in variants:
            vlow = v.lower()
            if vlow in low:
                hits.append((i, line, v, 'lexical'))
                break
            score = fuzz.partial_ratio(vlow, low)
            if score >= fuzzy_threshold:
                hits.append((i, line, v, f'fuzzy:{score}'))
                break
    return hits


def clip_text(text, max_chars=MAX_CONTEXT_CHARS):
    if text is None:
        return ''
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + ' ...'


def chunk_list(items, n):
    for i in range(0, len(items), n):
        yield items[i:i+n]


def parse_batch_results(content, n):
    fallback = [{'label': 'WEAK', 'rationale': 'LLM parse failed'} for _ in range(n)]
    if not content:
        return fallback

    try:
        parsed = json.loads(content)
        results = parsed.get('results', [])
        mapped = {int(r['idx']): r for r in results if isinstance(r, dict) and 'idx' in r}
        out = []
        for i in range(n):
            r = mapped.get(i)
            if not r:
                out.append({'label': 'WEAK', 'rationale': 'No label'})
                continue
            label = str(r.get('label', 'WEAK')).upper().strip()
            if label not in {'DIRECT', 'INDIRECT', 'NONE'}:
                label = 'WEAK'
            out.append({'label': label, 'rationale': str(r.get('rationale', ''))})
        return out
    except Exception:
        return fallback


def classify_with_model(concept, contexts, model_name, hint_labels=None):
    compact = []
    for i, ctx in enumerate(contexts):
        row = {'idx': i, 'context': clip_text(ctx)}
        if hint_labels and i < len(hint_labels):
            row['hint_label'] = hint_labels[i]
        compact.append(row)

    system_prompt = (
        'You are an evidence auditor. '
        'For each snippet, decide if the concept is DIRECT, INDIRECT, or NONE. '
        'Return strict JSON object with key "results": '
        '[{"idx":0,"label":"DIRECT|INDIRECT|NONE","rationale":"..."}]'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        f'Snippets JSON:\n{json.dumps(compact, ensure_ascii=False)}'
    )

    content = safe_chat_completion(
        model_name=model_name,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.1
    )

    return parse_batch_results(content, len(contexts))


def classify_hits_batch(concept, contexts):
    if not contexts:
        return []

    if not LLM_CALLS:
        return [{
            'label': 'WEAK',
            'rationale': 'LLM disabled',
            'label_pass1': 'WEAK',
            'rationale_pass1': 'LLM disabled',
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        } for _ in contexts]

    pass1 = classify_with_model(concept, contexts, MODEL_PASS1)
    out = []
    for r in pass1:
        out.append({
            'label': r.get('label', 'WEAK'),
            'rationale': r.get('rationale', ''),
            'label_pass1': r.get('label', 'WEAK'),
            'rationale_pass1': r.get('rationale', ''),
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        })

    if USE_ESCALATION:
        idxs = [i for i, r in enumerate(out) if r['label'] in ESCALATE_LABELS]
        if idxs:
            sub_contexts = [contexts[i] for i in idxs]
            hints = [out[i]['label_pass1'] for i in idxs]
            pass2 = classify_with_model(concept, sub_contexts, MODEL_PASS2, hint_labels=hints)

            for j, i in enumerate(idxs):
                r2 = pass2[j]
                out[i]['label_pass2'] = r2.get('label', 'WEAK')
                out[i]['rationale_pass2'] = r2.get('rationale', '')
                out[i]['model_pass2'] = MODEL_PASS2
                out[i]['escalated'] = True

                if r2.get('label') in {'DIRECT', 'INDIRECT', 'NONE'}:
                    out[i]['label'] = r2.get('label')
                    out[i]['rationale'] = r2.get('rationale', '')

    return out


def classify_hits_chunked(concept, contexts):
    out = []
    for chunk in chunk_list(contexts, CLASSIFY_CHUNK_SIZE):
        out.extend(classify_hits_batch(concept, chunk))
    return out


def append_rows_csv(out_csv, rows):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    if out_csv.exists():
        df_new.to_csv(out_csv, mode='a', header=False, index=False)
    else:
        df_new.to_csv(out_csv, index=False)


def checkpoint_path(section_name):
    return CHECKPOINT_DIR / f'{section_name}_done_ids.json'


def load_done_ids(section_name):
    if not RESUME:
        return set()
    cp = checkpoint_path(section_name)
    if not cp.exists():
        return set()
    try:
        data = json.loads(cp.read_text(encoding='utf-8'))
        return set(data)
    except Exception:
        return set()


def save_done_ids(section_name, done_ids):
    cp = checkpoint_path(section_name)
    cp.write_text(json.dumps(sorted(list(done_ids)), ensure_ascii=False, indent=2), encoding='utf-8')


def process_in_batches(records):
    for batch in chunk_list(records, BATCH_SIZE_PAPERS):
        yield batch


def llm_fields_from_cls(cls):
    return {
        'llm_model_pass1': cls.get('model_pass1', ''),
        'llm_label_pass1': cls.get('label_pass1', ''),
        'llm_model_pass2': cls.get('model_pass2', ''),
        'llm_label_pass2': cls.get('label_pass2', ''),
        'llm_escalated': cls.get('escalated', False),
    }


In [9]:
# @title 9. Section 7A Evidence (Smart Infrastructure)

def run_section7_domain_extraction(section_code, macro_domain, lexical_terms):
    section_name = f'section7{section_code}'
    out_csv = OUTPUT_DIR / f'{section_name}_evidence.csv'
    done_ids = load_done_ids(section_name)
    pending = [p for p in papers if p['paper_id'] not in done_ids]
    print(f'{section_name}: pending papers = {len(pending)}')

    variants = []
    seen = set()
    for term in lexical_terms:
        for item in get_variants(term):
            key = str(item).strip().lower()
            if key and key not in seen:
                seen.add(key)
                variants.append(str(item).strip())
    variants = variants[:MAX_VARIANTS_PER_CONCEPT]

    for batch in process_in_batches(pending):
        batch_rows = []
        for paper in tqdm(batch, desc=f'{section_name} batch'):
            paper_id = paper['paper_id']
            lines = paper['lines']
            heading_map = build_heading_map(lines)
            record = json_index.get(paper_id, {})
            medium = get_record_medium(record)
            app_bundle = get_application_bundle(record)

            hits = scan_lines_for_variants(lines, variants)[:MAX_HITS_PER_CONCEPT_PER_PAPER]
            contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
            cls_all = classify_hits_chunked(macro_domain, contexts)

            for (hit, cls) in zip(hits, cls_all):
                idx, line, variant, match_type = hit
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': f'7{section_code}',
                    'macro_domain': macro_domain,
                    'variant': variant,
                    'match_type': match_type,
                    'strength': cls.get('label', 'WEAK'),
                    'rationale': cls.get('rationale', ''),
                    'quote': line.strip(),
                    'line_start': idx + 1,
                    'line_end': idx + 1,
                    'heading_path': heading_map.get(idx, 'no_heading'),
                    'json_path': '',
                    'json_value': '',
                    'medium': medium,
                    'application_domains_json': ';'.join(app_bundle['raw_domains']),
                    'application_domains_canonical': ';'.join(app_bundle['canonical_domains']),
                    'macro_domains_json': ';'.join(app_bundle['macro_domains']),
                    'scenario_description': app_bundle['scenario_description'],
                    'scenario_labels': ';'.join(app_bundle['scenario_labels']),
                    **llm_fields_from_cls(cls),
                })

            if macro_domain in app_bundle['macro_domains']:
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': f'7{section_code}',
                    'macro_domain': macro_domain,
                    'variant': '',
                    'match_type': 'json',
                    'strength': 'DIRECT',
                    'rationale': 'Structured study-level application domain mapping',
                    'quote': app_bundle['scenario_description'],
                    'line_start': '',
                    'line_end': '',
                    'heading_path': '',
                    'json_path': 'study_level.application.application_domain',
                    'json_value': ';'.join(app_bundle['raw_domains']),
                    'medium': medium,
                    'application_domains_json': ';'.join(app_bundle['raw_domains']),
                    'application_domains_canonical': ';'.join(app_bundle['canonical_domains']),
                    'macro_domains_json': ';'.join(app_bundle['macro_domains']),
                    'scenario_description': app_bundle['scenario_description'],
                    'scenario_labels': ';'.join(app_bundle['scenario_labels']),
                    'llm_model_pass1': 'json',
                    'llm_label_pass1': 'DIRECT',
                    'llm_model_pass2': '',
                    'llm_label_pass2': '',
                    'llm_escalated': False,
                })

            done_ids.add(paper_id)

        append_rows_csv(out_csv, batch_rows)
        save_done_ids(section_name, done_ids)
        print(f'{section_name}: wrote {len(batch_rows)} rows; done={len(done_ids)}')

    print('Saved:', out_csv)


terms_7A = [
    'smart infrastructure', 'infrastructure monitoring', 'structural health monitoring',
    'pipeline monitoring', 'industrial monitoring', 'smart city', 'grid monitoring',
    'network monitoring', 'factory automation', 'distributed sensing'
]
run_section7_domain_extraction('A', 'smart_infrastructure', terms_7A)


section7A: pending papers = 221


section7A batch: 100%|██████████| 10/10 [00:25<00:00,  2.53s/it]


section7A: wrote 64 rows; done=10


section7A batch: 100%|██████████| 10/10 [00:24<00:00,  2.50s/it]


section7A: wrote 66 rows; done=20


...

section7A: wrote 64 rows; done=220


section7A batch:   0%|          | 0/1 [00:00<?, ?it/s]

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.3s
Rate limit guard (llama-3.3-70b-versatile): sleeping 0.3s


section7A batch: 100%|██████████| 1/1 [00:03<00:00,  3.69s/it]

section7A: wrote 6 rows; done=221
Saved: analysis/VII_ev_v2/section7A_evidence.csv


In [10]:
# @title 10. Section 7B Evidence (Indoor Environments)
terms_7B = [
    'indoor positioning', 'indoor localization', 'smart lighting', 'occupancy sensing',
    'indoor navigation', 'retail sensing', 'hospital monitoring', 'assisted living',
    'human computer interaction', 'asset tracking'
]
run_section7_domain_extraction('B', 'indoor_environments', terms_7B)


section7B: pending papers = 221


section7B batch: 100%|██████████| 10/10 [00:13<00:00,  1.38s/it]


section7B: wrote 32 rows; done=10


section7B batch: 100%|██████████| 10/10 [00:07<00:00,  1.29it/s]


section7B: wrote 17 rows; done=20


...

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.2s


section7B batch: 100%|██████████| 10/10 [00:10<00:00,  1.06s/it]


section7B: wrote 25 rows; done=220


section7B batch: 100%|██████████| 1/1 [00:01<00:00,  1.59s/it]

section7B: wrote 4 rows; done=221
Saved: analysis/VII_ev_v2/section7B_evidence.csv


In [11]:
# @title 11. Section 7C Evidence (Automotive and Transportation)
terms_7C = [
    'vehicular', 'vehicle to vehicle', 'vehicle to infrastructure', 'v2v', 'v2i',
    'autonomous vehicle', 'traffic monitoring', 'intelligent transportation',
    'automotive lidar communication', 'uav mobility sensing'
]
run_section7_domain_extraction('C', 'automotive_transportation', terms_7C)


section7C: pending papers = 221


section7C batch: 100%|██████████| 10/10 [00:22<00:00,  2.23s/it]


section7C: wrote 51 rows; done=10


section7C batch:  80%|████████  | 8/10 [00:13<00:02,  1.47s/it]

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.1s


...

section7C: wrote 63 rows; done=210


section7C batch: 100%|██████████| 10/10 [00:24<00:00,  2.42s/it]


section7C: wrote 48 rows; done=220


section7C batch: 100%|██████████| 1/1 [00:03<00:00,  3.56s/it]

section7C: wrote 6 rows; done=221
Saved: analysis/VII_ev_v2/section7C_evidence.csv


In [12]:
# @title 12. Section 7D Evidence (Underwater and Harsh Environments)
terms_7D = [
    'underwater optical communication', 'marine sensing', 'subsea monitoring',
    'oceanographic sensing', 'harsh environment', 'extreme environment',
    'volcanic ash detection', 'corrosive environment', 'offshore monitoring'
]
run_section7_domain_extraction('D', 'underwater_harsh', terms_7D)


section7D: pending papers = 221


section7D batch: 100%|██████████| 10/10 [00:05<00:00,  1.81it/s]


section7D: wrote 8 rows; done=10


section7D batch: 100%|██████████| 10/10 [00:14<00:00,  1.40s/it]


section7D: wrote 25 rows; done=20


...

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.2s


section7D batch: 100%|██████████| 10/10 [00:12<00:00,  1.21s/it]


section7D: wrote 16 rows; done=220


section7D batch: 100%|██████████| 1/1 [00:00<00:00, 62.07it/s]

section7D: wrote 0 rows; done=221
Saved: analysis/VII_ev_v2/section7D_evidence.csv


In [13]:
# @title 13. Section 7E Evidence + Section 7F Summary Outputs
terms_7E = [
    'satellite communication', 'inter-satellite link', 'space sensing',
    'space debris sensing', 'orbital monitoring', 'aerospace communication',
    'leo satellite', 'spaceborne sensing'
]
run_section7_domain_extraction('E', 'space_satellite', terms_7E)


a_csv = OUTPUT_DIR / 'section7A_evidence.csv'
b_csv = OUTPUT_DIR / 'section7B_evidence.csv'
c_csv = OUTPUT_DIR / 'section7C_evidence.csv'
d_csv = OUTPUT_DIR / 'section7D_evidence.csv'
e_csv = OUTPUT_DIR / 'section7E_evidence.csv'

A = pd.read_csv(a_csv) if a_csv.exists() else pd.DataFrame()
B = pd.read_csv(b_csv) if b_csv.exists() else pd.DataFrame()
C = pd.read_csv(c_csv) if c_csv.exists() else pd.DataFrame()
D = pd.read_csv(d_csv) if d_csv.exists() else pd.DataFrame()
E = pd.read_csv(e_csv) if e_csv.exists() else pd.DataFrame()

macro_to_df = {
    'smart_infrastructure': A,
    'indoor_environments': B,
    'automotive_transportation': C,
    'underwater_harsh': D,
    'space_satellite': E,
}


def strict_supported_ids(df):
    if df.empty or 'paper_id' not in df.columns:
        return set()
    out = set()
    for pid, grp in df.groupby('paper_id'):
        direct = (grp['strength'].astype(str).str.upper() == 'DIRECT').sum() if 'strength' in grp.columns else 0
        indirect = (grp['strength'].astype(str).str.upper() == 'INDIRECT').sum() if 'strength' in grp.columns else 0
        if direct >= 1 or indirect >= 2:
            out.add(str(pid))
    return out


def json_supported_ids(df):
    if df.empty or 'paper_id' not in df.columns or 'match_type' not in df.columns:
        return set()
    sub = df[df['match_type'].astype(str).str.lower() == 'json']
    return set(sub['paper_id'].astype(str)) if not sub.empty else set()


all_paper_ids = sorted([p.get('paper_id') for p in papers])
macro_supported = {macro: strict_supported_ids(df) for macro, df in macro_to_df.items()}
macro_json = {macro: json_supported_ids(df) for macro, df in macro_to_df.items()}

paper_rows = []
for pid in all_paper_ids:
    rec = json_index.get(pid, {})
    medium = get_record_medium(rec)
    app_bundle = get_application_bundle(rec)
    row = {
        'paper_id': pid,
        'medium': medium,
        'application_domains_json': ';'.join(app_bundle['raw_domains']),
        'application_domains_canonical': ';'.join(app_bundle['canonical_domains']),
        'macro_domains_json': ';'.join(app_bundle['macro_domains']),
    }
    count_macro = 0
    for macro in macro_to_df.keys():
        has_val = pid in macro_supported[macro]
        row[f'has_{macro}'] = has_val
        if has_val:
            count_macro += 1
    row['n_supported_macro_domains'] = count_macro
    paper_rows.append(row)

paper_df = pd.DataFrame(paper_rows)
paper_map_csv = OUTPUT_DIR / 'section7F_paper_macro_map.csv'
paper_df.to_csv(paper_map_csv, index=False)

summary_row = {
    'n_total_papers': int(len(all_paper_ids)),
    'n_smart_infrastructure_papers': int(len(macro_supported['smart_infrastructure'])),
    'n_indoor_environments_papers': int(len(macro_supported['indoor_environments'])),
    'n_automotive_transportation_papers': int(len(macro_supported['automotive_transportation'])),
    'n_underwater_harsh_papers': int(len(macro_supported['underwater_harsh'])),
    'n_space_satellite_papers': int(len(macro_supported['space_satellite'])),
    'n_multi_macro_domain_papers': int((paper_df['n_supported_macro_domains'] >= 2).sum()) if not paper_df.empty else 0,
}
summary_table_csv = OUTPUT_DIR / 's7f_app_sum_tbl.csv'
pd.DataFrame([summary_row]).to_csv(summary_table_csv, index=False)

transfer_rows = []
for macro, ids in macro_supported.items():
    sub = paper_df[paper_df['paper_id'].isin(ids)] if not paper_df.empty else pd.DataFrame()
    if sub.empty:
        continue
    by_medium = sub.groupby('medium', dropna=False).agg(n_papers=('paper_id', 'nunique')).reset_index()
    for _, r in by_medium.iterrows():
        transfer_rows.append({'macro_domain': macro, 'medium': str(r['medium']), 'n_papers': int(r['n_papers'])})

transfer_df = pd.DataFrame(transfer_rows)
transfer_csv = OUTPUT_DIR / 'section7F_transfer_map.csv'
transfer_df.to_csv(transfer_csv, index=False)

coverage_rows = []
for macro, ids in macro_supported.items():
    sub = transfer_df[transfer_df['macro_domain'] == macro] if not transfer_df.empty else pd.DataFrame()
    coverage_rows.append({
        'macro_domain': macro,
        'n_papers': int(len(ids)),
        'n_mediums': int(sub['medium'].nunique()) if not sub.empty else 0,
        'n_json_flag_papers': int(len(macro_json[macro])),
    })
coverage_df = pd.DataFrame(coverage_rows)
coverage_csv = OUTPUT_DIR / 's7f_macro_med_cov.csv'
coverage_df.to_csv(coverage_csv, index=False)

micro_counter = {}
for pid in all_paper_ids:
    rec = json_index.get(pid, {})
    for d in get_application_domains_canonical(rec):
        micro_counter[d] = micro_counter.get(d, 0) + 1

micro_df = pd.DataFrame([
    {'application_domain': k, 'n_papers': v} for k, v in sorted(micro_counter.items(), key=lambda kv: (-kv[1], kv[0]))
])
micro_csv = OUTPUT_DIR / 's7f_micro_dom_cnts.csv'
micro_df.to_csv(micro_csv, index=False)

summary_payload = dict(summary_row)
summary_payload['n_macro_domains_ge2_mediums'] = int((coverage_df['n_mediums'] >= 2).sum()) if not coverage_df.empty else 0
summary_payload['n_macro_domains_ge3_mediums'] = int((coverage_df['n_mediums'] >= 3).sum()) if not coverage_df.empty else 0
summary_payload['n_unique_micro_domains'] = int(len(micro_df))

summary_json = OUTPUT_DIR / 'section7F_summary.json'
summary_json.write_text(json.dumps(summary_payload, indent=2), encoding='utf-8')

print('Saved:', summary_table_csv)
print('Saved:', transfer_csv)
print('Saved:', coverage_csv)
print('Saved:', micro_csv)
print('Saved:', paper_map_csv)
print('Saved:', summary_json)


section7E: pending papers = 221


section7E batch: 100%|██████████| 10/10 [00:07<00:00,  1.28it/s]


section7E: wrote 14 rows; done=10


section7E batch:  80%|████████  | 8/10 [00:03<00:00,  2.54it/s]

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.1s
Rate limit guard (llama-3.3-70b-versatile): sleeping 1.1s


...

Rate limit guard (llama-3.3-70b-versatile): sleeping 1.1s


section7E batch: 100%|██████████| 10/10 [00:15<00:00,  1.54s/it]


section7E: wrote 22 rows; done=220


section7E batch: 100%|██████████| 1/1 [00:00<00:00, 61.81it/s]


section7E: wrote 0 rows; done=221
Saved: analysis/VII_ev_v2/section7E_evidence.csv
Saved: analysis/VII_ev_v2/s7f_app_sum_tbl.csv
Saved: analysis/VII_ev_v2/section7F_transfer_map.csv
Saved: analysis/VII_ev_v2/s7f_macro_med_cov.csv
Saved: analysis/VII_ev_v2/s7f_micro_dom_cnts.csv
Saved: analysis/VII_ev_v2/section7F_paper_macro_map.csv
Saved: analysis/VII_ev_v2/section7F_summary.json


In [14]:
# @title 14. Post-Processing Artifacts (Section 7)
import hashlib
from collections import defaultdict


def as_int_or_blank(x):
    try:
        if pd.isna(x) or x == '':
            return ''
        return int(float(x))
    except Exception:
        return ''


a_csv = OUTPUT_DIR / 'section7A_evidence.csv'
b_csv = OUTPUT_DIR / 'section7B_evidence.csv'
c_csv = OUTPUT_DIR / 'section7C_evidence.csv'
d_csv = OUTPUT_DIR / 'section7D_evidence.csv'
e_csv = OUTPUT_DIR / 'section7E_evidence.csv'

A = pd.read_csv(a_csv) if a_csv.exists() else pd.DataFrame()
B = pd.read_csv(b_csv) if b_csv.exists() else pd.DataFrame()
C = pd.read_csv(c_csv) if c_csv.exists() else pd.DataFrame()
D = pd.read_csv(d_csv) if d_csv.exists() else pd.DataFrame()
E = pd.read_csv(e_csv) if e_csv.exists() else pd.DataFrame()
frames = [('7A', A), ('7B', B), ('7C', C), ('7D', D), ('7E', E)]

# retrieval_hits.jsonl
retrieval_path = OUTPUT_DIR / 'retrieval_hits.jsonl'
with retrieval_path.open('w', encoding='utf-8') as f:
    for sec_name, df in frames:
        if df.empty:
            continue
        for _, r in df.iterrows():
            mt = str(r.get('match_type', ''))
            if mt.lower() == 'json':
                continue
            rec = {
                'paper_id': str(r.get('paper_id', '')),
                'section': sec_name,
                'macro_domain': str(r.get('macro_domain', '')),
                'variant': str(r.get('variant', '')),
                'match_type': mt,
                'quote': str(r.get('quote', '')),
                'line_start': as_int_or_blank(r.get('line_start', '')),
                'line_end': as_int_or_blank(r.get('line_end', '')),
                'heading_path': str(r.get('heading_path', '')),
                'strength': str(r.get('strength', '')),
                'rationale': str(r.get('rationale', '')),
            }
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('Saved:', retrieval_path)

# anchor_table.csv
anchor_rows = []
for sec_name, df in frames:
    if df.empty:
        continue
    for _, r in df.iterrows():
        macro = str(r.get('macro_domain', '')).strip()
        claim_key = f'{sec_name}|{macro}'
        claim_id = hashlib.sha1(claim_key.encode('utf-8')).hexdigest()[:12]
        anchor_rows.append({
            'claim_id': claim_id,
            'claim_key': claim_key,
            'section': sec_name,
            'paper_id': str(r.get('paper_id', '')),
            'macro_domain': macro,
            'variant': str(r.get('variant', '')),
            'match_type': str(r.get('match_type', '')),
            'strength': str(r.get('strength', '')),
            'rationale': str(r.get('rationale', '')),
            'quote': str(r.get('quote', '')),
            'line_start': as_int_or_blank(r.get('line_start', '')),
            'line_end': as_int_or_blank(r.get('line_end', '')),
            'heading_path': str(r.get('heading_path', '')),
            'json_path': str(r.get('json_path', '')),
            'json_value': str(r.get('json_value', '')),
        })

anchor_cols = ['claim_id','claim_key','section','paper_id','macro_domain','variant','match_type','strength','rationale','quote','line_start','line_end','heading_path','json_path','json_value','claim_supported']
if anchor_rows:
    anc = pd.DataFrame(anchor_rows)
    agg = anc.assign(
        is_direct=anc['strength'].astype(str).str.upper().eq('DIRECT'),
        is_indirect=anc['strength'].astype(str).str.upper().eq('INDIRECT')
    ).groupby('claim_id', as_index=False)[['is_direct', 'is_indirect']].sum()
    agg['claim_supported'] = (agg['is_direct'] >= 1) | (agg['is_indirect'] >= 2)
    anc = anc.merge(agg[['claim_id', 'claim_supported']], on='claim_id', how='left')
else:
    anc = pd.DataFrame(columns=anchor_cols)
anc = anc[anchor_cols]
anchor_path = OUTPUT_DIR / 'anchor_table.csv'
anc.to_csv(anchor_path, index=False)
print('Saved:', anchor_path)

# evidence_graph.jsonl + cluster_map.csv
supported_by_paper = defaultdict(set)
if not anc.empty:
    for _, r in anc.iterrows():
        if bool(r.get('claim_supported', False)):
            supported_by_paper[str(r.get('paper_id', ''))].add(str(r.get('macro_domain', '')))

anchors_by_paper = defaultdict(list)
for _, r in anc.iterrows():
    anchors_by_paper[str(r.get('paper_id', ''))].append({'section': str(r.get('section', '')), 'macro_domain': str(r.get('macro_domain', '')), 'strength': str(r.get('strength', ''))})

graph_path = OUTPUT_DIR / 'evidence_graph.jsonl'
cluster_rows = []
with graph_path.open('w', encoding='utf-8') as f:
    for paper_id, rec in sorted(json_index.items()):
        medium = get_record_medium(rec)
        app_bundle = get_application_bundle(rec)
        concepts = supported_by_paper.get(paper_id, set())
        graph_rec = {
            'paper_id': paper_id,
            'structured': {
                'medium': medium,
                'application_domains_raw': app_bundle['raw_domains'],
                'application_domains_canonical': app_bundle['canonical_domains'],
                'macro_domains_json': app_bundle['macro_domains'],
                'has_smart_infrastructure_supported': 'smart_infrastructure' in concepts,
                'has_indoor_environments_supported': 'indoor_environments' in concepts,
                'has_automotive_transportation_supported': 'automotive_transportation' in concepts,
                'has_underwater_harsh_supported': 'underwater_harsh' in concepts,
                'has_space_satellite_supported': 'space_satellite' in concepts,
            },
            'anchor_count': len(anchors_by_paper.get(paper_id, [])),
            'anchors': anchors_by_paper.get(paper_id, []),
        }
        f.write(json.dumps(graph_rec, ensure_ascii=False) + '\n')
        cluster_rows.append({
            'paper_id': paper_id,
            'medium': medium,
            'macro_domains_json': ';'.join(app_bundle['macro_domains']),
            'has_smart_infrastructure_supported': graph_rec['structured']['has_smart_infrastructure_supported'],
            'has_indoor_environments_supported': graph_rec['structured']['has_indoor_environments_supported'],
            'has_automotive_transportation_supported': graph_rec['structured']['has_automotive_transportation_supported'],
            'has_underwater_harsh_supported': graph_rec['structured']['has_underwater_harsh_supported'],
            'has_space_satellite_supported': graph_rec['structured']['has_space_satellite_supported'],
            'anchor_count': graph_rec['anchor_count'],
            'confidence': 'high' if graph_rec['anchor_count'] >= 8 else ('medium' if graph_rec['anchor_count'] >= 3 else 'low'),
        })
cluster_path = OUTPUT_DIR / 'cluster_map.csv'
pd.DataFrame(cluster_rows).to_csv(cluster_path, index=False)
print('Saved:', graph_path)
print('Saved:', cluster_path)

# axis + mapping docs
axis_md = '\n'.join([
    '# Section 7 Axis Definitions (v2)',
    '',
    'Axis-1 Medium: normalized labels aligned with Section IV taxonomy mapping.',
    'Axis-2 Application macro domains: smart_infrastructure, indoor_environments, automotive_transportation, underwater_harsh, space_satellite.',
    'Axis-3 Application metadata: study-level domain tags + scenario description + scenario labels.',
    'Axis-4 Evidence gate: claim_supported = (>=1 DIRECT) OR (>=2 INDIRECT).',
    'Governance note: Section VII follows Section II metric-plane guardrails for narrative claims.',
])
(OUTPUT_DIR / 'axis_definitions.md').write_text(axis_md, encoding='utf-8')

mapping_md = '\n'.join([
    '# Section 7 Mapping Rules (v2)',
    '',
    '1. study_level.application.application_domain is the primary structured source.',
    '2. scenario_description + scenario_label provide secondary disambiguation context.',
    '3. Text anchors complement structured domains for application narrative coverage.',
    '4. Medium normalization must stay consistent with Section IV outputs.',
    '5. Section VI enabler mentions are contextual and cannot replace application-domain evidence.',
    '6. Section II plane-separation language applies when SNR/OSNR appears in application claims.',
])
(OUTPUT_DIR / 'mapping_rules.md').write_text(mapping_md, encoding='utf-8')
print('Saved:', OUTPUT_DIR / 'axis_definitions.md')
print('Saved:', OUTPUT_DIR / 'mapping_rules.md')

# contract_violations.csv
violations = []
seen = set()
for paper_id, rec in sorted(json_index.items()):
    medium = get_record_medium(rec)
    app_bundle = get_application_bundle(rec)
    if not app_bundle['raw_domains']:
        key = (paper_id, 'APP_MISSING')
        if key not in seen:
            seen.add(key)
            violations.append({'paper_id': paper_id, 'section': '7*', 'category': 'APP_MISSING', 'severity': 'MAJOR', 'reason': 'study_level.application.application_domain is missing', 'evidence': 'application_domain=missing'})
    if medium == 'unknown':
        key = (paper_id, 'MEDIUM_UNKNOWN')
        if key not in seen:
            seen.add(key)
            violations.append({'paper_id': paper_id, 'section': '7*', 'category': 'MEDIUM_UNKNOWN', 'severity': 'MINOR', 'reason': 'Medium label is unknown', 'evidence': 'oisac_medium_class=unknown'})

for sec_name, df in frames:
    if df.empty:
        continue
    text_df = df[df['match_type'].astype(str).str.lower() != 'json'].copy() if 'match_type' in df.columns else df.copy()
    for pid, grp in text_df.groupby('paper_id'):
        direct = (grp['strength'].astype(str).str.upper() == 'DIRECT').sum() if 'strength' in grp.columns else 0
        indirect = (grp['strength'].astype(str).str.upper() == 'INDIRECT').sum() if 'strength' in grp.columns else 0
        if direct < 1 and indirect < 2:
            macro = str(grp['macro_domain'].iloc[0]) if 'macro_domain' in grp.columns else 'unknown'
            key = (str(pid), 'EVIDENCE_WEAK', macro)
            if key not in seen:
                seen.add(key)
                violations.append({'paper_id': str(pid), 'section': sec_name, 'category': 'EVIDENCE_WEAK', 'severity': 'MINOR', 'reason': f'{macro} lacks support gate (text anchors)', 'evidence': f'direct={direct}; indirect={indirect}'})

    for _, r in text_df.iterrows():
        pid = str(r.get('paper_id', ''))
        macro = str(r.get('macro_domain', ''))
        heading = str(r.get('heading_path', '')).lower()
        strength = str(r.get('strength', '')).upper()
        quote = str(r.get('quote', '')).lower()
        macro_json = set([x for x in str(r.get('macro_domains_json', '')).split(';') if x])
        if 'reference' in heading and strength in {'DIRECT', 'INDIRECT'}:
            key = (pid, 'REFERENCE_NOISE', macro)
            if key not in seen:
                seen.add(key)
                violations.append({'paper_id': pid, 'section': sec_name, 'category': 'REFERENCE_NOISE', 'severity': 'MINOR', 'reason': 'Anchor under reference-like heading', 'evidence': f'heading_path={heading}'})
        if macro_json and macro not in macro_json and strength == 'DIRECT':
            key = (pid, 'APP_SCOPE_MISMATCH', macro)
            if key not in seen:
                seen.add(key)
                violations.append({'paper_id': pid, 'section': sec_name, 'category': 'APP_SCOPE_MISMATCH', 'severity': 'MAJOR', 'reason': 'DIRECT text claim conflicts with structured macro tags', 'evidence': f'text_macro={macro}; json_macros={";".join(sorted(list(macro_json)))}'})
        if 'osnr' in quote and 'snr' in quote:
            key = (pid, 'PLANE_MIX_MENTION', sec_name)
            if key not in seen:
                seen.add(key)
                violations.append({'paper_id': pid, 'section': sec_name, 'category': 'PLANE_MIX_MENTION', 'severity': 'MINOR', 'reason': 'Quote contains both OSNR and SNR terms; verify plane wording', 'evidence': str(r.get('quote', ''))[:200]})

viol_df = pd.DataFrame(violations, columns=['paper_id', 'section', 'category', 'severity', 'reason', 'evidence'])
viol_path = OUTPUT_DIR / 'contract_violations.csv'
viol_df.to_csv(viol_path, index=False)
print('Saved:', viol_path, 'rows=', len(viol_df))


Saved: analysis/VII_ev_v2/retrieval_hits.jsonl
Saved: analysis/VII_ev_v2/anchor_table.csv
Saved: analysis/VII_ev_v2/evidence_graph.jsonl
Saved: analysis/VII_ev_v2/cluster_map.csv
Saved: analysis/VII_ev_v2/axis_definitions.md
Saved: analysis/VII_ev_v2/mapping_rules.md
Saved: analysis/VII_ev_v2/contract_violations.csv rows= 607


In [15]:
# @title 15. Section 7G Dual-View Comparison (JSON vs Evidence)
macro_files = {
    'smart_infrastructure': OUTPUT_DIR / 'section7A_evidence.csv',
    'indoor_environments': OUTPUT_DIR / 'section7B_evidence.csv',
    'automotive_transportation': OUTPUT_DIR / 'section7C_evidence.csv',
    'underwater_harsh': OUTPUT_DIR / 'section7D_evidence.csv',
    'space_satellite': OUTPUT_DIR / 'section7E_evidence.csv',
}


def strict_supported_ids(df):
    if df.empty or 'paper_id' not in df.columns:
        return set()
    out = set()
    for pid, grp in df.groupby('paper_id'):
        direct = (grp['strength'].astype(str).str.upper() == 'DIRECT').sum() if 'strength' in grp.columns else 0
        indirect = (grp['strength'].astype(str).str.upper() == 'INDIRECT').sum() if 'strength' in grp.columns else 0
        if direct >= 1 or indirect >= 2:
            out.add(str(pid))
    return out


json_flags = {k: set() for k in macro_files.keys()}
for pid, rec in sorted(json_index.items()):
    macros = set(get_macro_domains_from_json(rec))
    for macro in macro_files.keys():
        if macro in macros:
            json_flags[macro].add(pid)

rows = []
example_rows = []
max_show = 30
for macro, path in macro_files.items():
    df = pd.read_csv(path) if path.exists() else pd.DataFrame()
    raw_ids = set(df['paper_id'].astype(str)) if not df.empty and 'paper_id' in df.columns else set()
    strict_ids = strict_supported_ids(df)
    json_ids = json_flags[macro]

    rows.append({
        'macro_domain': macro,
        'study_flag_count': len(json_ids),
        'raw_evidence_count': len(raw_ids),
        'strict_evidence_count': len(strict_ids),
        'flag_intersection_raw': len(json_ids & raw_ids),
        'flag_intersection_strict': len(json_ids & strict_ids),
        'raw_only_vs_flag': len(raw_ids - json_ids),
        'strict_only_vs_flag': len(strict_ids - json_ids),
    })

    example_rows.append({'macro_domain': macro, 'group': 'flag_only', 'paper_ids': ';'.join(sorted(list(json_ids - raw_ids))[:max_show])})
    example_rows.append({'macro_domain': macro, 'group': 'raw_only', 'paper_ids': ';'.join(sorted(list(raw_ids - json_ids))[:max_show])})
    example_rows.append({'macro_domain': macro, 'group': 'strict_only', 'paper_ids': ';'.join(sorted(list(strict_ids - json_ids))[:max_show])})

cmp_df = pd.DataFrame(rows)
cmp_csv = OUTPUT_DIR / 's7g_dual_view_cmp.csv'
cmp_df.to_csv(cmp_csv, index=False)

examples_csv = OUTPUT_DIR / 's7g_dual_view_ex.csv'
pd.DataFrame(example_rows).to_csv(examples_csv, index=False)

md_lines = []
md_lines.append('# Section 7G Dual-View Comparison')
md_lines.append('')
md_lines.append('This report compares structured application tags (JSON) vs extracted evidence rows (raw and strict).')
md_lines.append('It is additive and does not overwrite Section VII outputs.')
md_lines.append('')
for _, r in cmp_df.iterrows():
    md_lines.append(f"## {r['macro_domain']}")
    md_lines.append(f"- study_flag_count: {int(r['study_flag_count'])}")
    md_lines.append(f"- raw_evidence_count: {int(r['raw_evidence_count'])}")
    md_lines.append(f"- strict_evidence_count: {int(r['strict_evidence_count'])}")
    md_lines.append(f"- flag_intersection_raw: {int(r['flag_intersection_raw'])}")
    md_lines.append(f"- flag_intersection_strict: {int(r['flag_intersection_strict'])}")
    md_lines.append(f"- raw_only_vs_flag: {int(r['raw_only_vs_flag'])}")
    md_lines.append(f"- strict_only_vs_flag: {int(r['strict_only_vs_flag'])}")
    md_lines.append('')

report_md = OUTPUT_DIR / 'section7G_dual_view_report.md'
report_md.write_text('\n'.join(md_lines), encoding='utf-8')

print('Saved:', cmp_csv)
print('Saved:', examples_csv)
print('Saved:', report_md)
print(cmp_df.to_string(index=False))


Saved: analysis/VII_ev_v2/s7g_dual_view_cmp.csv
Saved: analysis/VII_ev_v2/s7g_dual_view_ex.csv
Saved: analysis/VII_ev_v2/section7G_dual_view_report.md
             macro_domain  study_flag_count  raw_evidence_count  strict_evidence_count  flag_intersection_raw  flag_intersection_strict  raw_only_vs_flag  strict_only_vs_flag
     smart_infrastructure               103                 221                    204                    103                       103               118                  101
      indoor_environments                65                 175                     81                     65                        65               110                   16
automotive_transportation                76                 213                    104                     76                        76               137                   28
         underwater_harsh                16                 123                     23                     16                        16              

In [16]:
# @title 16. Readiness Report
report_files = [
    'section7A_evidence.csv',
    'section7B_evidence.csv',
    'section7C_evidence.csv',
    'section7D_evidence.csv',
    'section7E_evidence.csv',
    's7f_app_sum_tbl.csv',
    'section7F_summary.json',
    'section7F_transfer_map.csv',
    's7f_macro_med_cov.csv',
    's7f_micro_dom_cnts.csv',
    'section7F_paper_macro_map.csv',
    'evidence_graph.jsonl',
    'retrieval_hits.jsonl',
    'anchor_table.csv',
    'axis_definitions.md',
    'mapping_rules.md',
    'cluster_map.csv',
    'contract_violations.csv',
    's7g_dual_view_cmp.csv',
    's7g_dual_view_ex.csv',
    'section7G_dual_view_report.md',
]

report = []
for fname in report_files:
    p = OUTPUT_DIR / fname
    report.append(f"{fname}: " + ('OK' if p.exists() else 'MISSING'))

stats = []
try:
    for sec in ['A','B','C','D','E']:
        p = OUTPUT_DIR / f'section7{sec}_evidence.csv'
        if p.exists():
            d = pd.read_csv(p)
            stats.append(f'section7{sec}_rows: {len(d)}')
            stats.append(f'section7{sec}_unique_papers: {d["paper_id"].nunique() if "paper_id" in d.columns else 0}')

    ps = OUTPUT_DIR / 'section7F_summary.json'
    if ps.exists():
        s = json.loads(ps.read_text(encoding='utf-8'))
        for k in ['n_total_papers','n_smart_infrastructure_papers','n_indoor_environments_papers','n_automotive_transportation_papers','n_underwater_harsh_papers','n_space_satellite_papers','n_multi_macro_domain_papers','n_macro_domains_ge2_mediums','n_macro_domains_ge3_mediums','n_unique_micro_domains']:
            stats.append(f'{k}: {s.get(k)}')

    pvc = OUTPUT_DIR / 'contract_violations.csv'
    if pvc.exists():
        v = pd.read_csv(pvc)
        stats.append(f'contract_violations_rows: {len(v)}')

    pg = OUTPUT_DIR / 's7g_dual_view_cmp.csv'
    if pg.exists():
        g = pd.read_csv(pg)
        for _, row in g.iterrows():
            m = str(row.get('macro_domain', 'UNK'))
            stats.append(f'dual_{m}_study_flag_count: {int(row.get("study_flag_count", 0))}')
            stats.append(f'dual_{m}_raw_evidence_count: {int(row.get("raw_evidence_count", 0))}')
            stats.append(f'dual_{m}_strict_evidence_count: {int(row.get("strict_evidence_count", 0))}')
except Exception as e:
    stats.append(f'stats_error: {e}')

report_path = OUTPUT_DIR / 'readiness_report.md'
report_path.write_text('\n'.join(report + [''] + stats), encoding='utf-8')
print('\n'.join(report + [''] + stats))
print('Saved:', report_path)


section7A_evidence.csv: OK
section7B_evidence.csv: OK
section7C_evidence.csv: OK
section7D_evidence.csv: OK
section7E_evidence.csv: OK
s7f_app_sum_tbl.csv: OK
section7F_summary.json: OK
section7F_transfer_map.csv: OK
s7f_macro_med_cov.csv: OK
s7f_micro_dom_cnts.csv: OK
section7F_paper_macro_map.csv: OK
evidence_graph.jsonl: OK
retrieval_hits.jsonl: OK
anchor_table.csv: OK
axis_definitions.md: OK
mapping_rules.md: OK
cluster_map.csv: OK
contract_violations.csv: OK
s7g_dual_view_cmp.csv: OK
s7g_dual_view_ex.csv: OK
section7G_dual_view_report.md: OK

section7A_rows: 1426
section7A_unique_papers: 221
section7B_rows: 636
section7B_unique_papers: 175
section7C_rows: 1157
section7C_unique_papers: 213
section7D_rows: 298
section7D_unique_papers: 123
section7E_rows: 329
section7E_unique_papers: 135
n_total_papers: 221
n_smart_infrastructure_papers: 204
n_indoor_environments_papers: 81
n_automotive_transportation_papers: 104
n_underwater_harsh_papers: 23
n_space_satellite_papers: 34
n_multi_macr